In [ ]:
!pip install diffusers transformers accelerate safetensors datasets

!pip install diffusers

# xuat api
!pip install flask flask-cors pyngrok flask-ngrok

In [ ]:
from diffusers import DiffusionPipeline
import torch
import os

# ═══════════════════════════════════════════════════════════════
# MODEL CONFIGURATION - CHANGE THIS TO SWITCH MODELS
# ═══════════════════════════════════════════════════════════════
# Options: "sd-v1.5", "xl-base-0.9"
SELECTED_MODEL = "xl-base-0.9"  # Change to switch models
# ═══════════════════════════════════════════════════════════════

# Model configurations
MODEL_CONFIG = {
    "sd-v1.5": {
        "model_id": "runwayml/stable-diffusion-v1-5",
        "torch_dtype": torch.float16,
        "device": "cuda",
        "inference_steps": 50,
        "description": "Stable Diffusion v1.5 (faster, lower memory)"
    },
    "xl-base-0.9": {
        "model_id": "stabilityai/stable-diffusion-xl-base-0.9",
        "torch_dtype": torch.float16,
        "device": "cuda",
        "inference_steps": 50,
        "description": "Stable Diffusion 2.1 (good balance)"
    }
}

# Get selected model config
config = MODEL_CONFIG[SELECTED_MODEL]

print(f"\n🎨 Loading Model: {SELECTED_MODEL}")
print(f"   {config['description']}")
print(f"   Model ID: {config['model_id']}")
print(f"   Default Inference Steps: {config['inference_steps']}")
print(f"   Device: {config['device']}")
print("   This may take a few minutes on first load...\n")

# Load the selected model
pipe = DiffusionPipeline.from_pretrained(
    config["model_id"], 
    torch_dtype=config["torch_dtype"]
)
pipe.to(config["device"])
pipe.safety_checker = None  # Tắt bộ lọc NSFW

print(f"✓ Model loaded successfully!\n")

# Store model info globally for API endpoints
MODEL_INFO = {
    "name": SELECTED_MODEL,
    "config": config,
    "pipe": pipe
}

In [ ]:
! ngrok authtoken '' # copy paste từ env
# API KEY cho imgbb
IMGBB_API_KEY = "" # copy paste từ env

import threading
import base64
import requests
from io import BytesIO
from flask import Flask, request, jsonify
from pyngrok import ngrok
from flask_cors import CORS
import torch
import requests
import json

# Khởi tạo ứng dụng Flask
app = Flask(__name__)
CORS(app)  # Cho phép tất cả các nguồn truy cập API

def upload_to_imgbb(image):
    """
    Upload ảnh lên imgbb và trả về URL
    Args:
        image: PIL Image object
    Returns:
        str: URL của ảnh trên imgbb hoặc None nếu thất bại
    """
    try:
        # Chuyển ảnh thành base64
        buffered = BytesIO()
        image.save(buffered, format="PNG")
        img_base64 = base64.b64encode(buffered.getvalue()).decode('utf-8')
        
        # Upload lên imgbb
        url = "https://api.imgbb.com/1/upload"
        payload = {
            "key": IMGBB_API_KEY,
            "image": img_base64
        }
        
        response = requests.post(url, data=payload)
        data = response.json()
        
        if data.get("success"):
            return data["data"]["url"]
        else:
            print(f"imgbb upload failed: {data}")
            return None
    
    except Exception as e:
        print(f"Error uploading to imgbb: {str(e)}")
        return None

@app.route('/health', methods=['GET'])
def health():
    """Kiểm tra sức khỏe API"""
    return jsonify({
        "status": "OK", 
        "message": "API is running",
        "model": MODEL_INFO["name"],
        "model_description": MODEL_INFO["config"]["description"]
    })

@app.route('/models', methods=['GET'])
def list_models():
    """Liệt kê các model có sẵn"""
    models = []
    for model_key, config in MODEL_CONFIG.items():
        models.append({
            "id": model_key,
            "name": config["description"],
            "inference_steps": config["inference_steps"],
            "current": model_key == MODEL_INFO["name"]
        })
    return jsonify({
        "available_models": models,
        "current_model": MODEL_INFO["name"]
    })

@app.route('/generate-image', methods=['POST'])
def generate_image():
    """
    Endpoint để sinh ảnh từ prompt
    Request body: {
        "prompt": "your prompt here", 
        "num_inference_steps": 50  (optional, uses model default if not provided)
    }
    Response: {"success": true, "image_url": "imgbb_url", "message": "..."}
    """
    try:
        data = request.get_json()
        
        # Kiểm tra prompt
        if 'prompt' not in data or not data['prompt']:
            return jsonify({"success": False, "error": "Missing or empty 'prompt' field"}), 400
        
        prompt = data['prompt']
        # Sử dụng default inference steps từ config model nếu không được cung cấp
        num_inference_steps = data.get('num_inference_steps', MODEL_INFO["config"]["inference_steps"])
        
        print(f"\n🎨 Generating image for prompt: {prompt}")
        print(f"   Model: {MODEL_INFO['name']}")
        print(f"   Inference steps: {num_inference_steps}")
        
        # Sinh ảnh
        with torch.no_grad():
            image = pipe(prompt, num_inference_steps=num_inference_steps).images[0]
        
        # Upload lên imgbb
        print("   Uploading to imgbb...")
        image_url = upload_to_imgbb(image)
        
        if image_url:
            return jsonify({
                "success": True,
                "image_url": image_url,
                "prompt": prompt,
                "model": MODEL_INFO["name"],
                "inference_steps": num_inference_steps,
                "message": "Image generated and uploaded successfully"
            })
        else:
            return jsonify({
                "success": False,
                "error": "Failed to upload image to imgbb"
            }), 500
    
    except Exception as e:
        print(f"Error: {str(e)}")
        return jsonify({
            "success": False,
            "error": f"Failed to generate image: {str(e)}"
        }), 500

@app.route('/generate-batch', methods=['POST'])
def generate_batch():
    """
    Endpoint để sinh nhiều ảnh từ các prompt khác nhau
    Request body: {
        "prompts": ["prompt1", "prompt2"], 
        "num_inference_steps": 50  (optional)
    }
    Response: {"success": true, "images": [{"prompt": "...", "image_url": "..."}]}
    """
    try:
        data = request.get_json()
        
        if 'prompts' not in data or not isinstance(data['prompts'], list):
            return jsonify({"success": False, "error": "Missing or invalid 'prompts' field"}), 400
        
        prompts = data['prompts']
        num_inference_steps = data.get('num_inference_steps', MODEL_INFO["config"]["inference_steps"])
        
        images = []
        for i, prompt in enumerate(prompts):
            print(f"\n   Generating image {i+1}/{len(prompts)}: {prompt}")
            with torch.no_grad():
                image = pipe(prompt, num_inference_steps=num_inference_steps).images[0]
            
            # Upload lên imgbb
            print(f"   Uploading image {i+1}/{len(prompts)} to imgbb...")
            image_url = upload_to_imgbb(image)
            
            if image_url:
                images.append({
                    "prompt": prompt,
                    "image_url": image_url
                })
            else:
                print(f"   Failed to upload image for prompt: {prompt}")
        
        if images:
            return jsonify({
                "success": True,
                "images": images,
                "model": MODEL_INFO["name"],
                "total": len(images),
                "inference_steps": num_inference_steps,
                "message": f"Generated {len(images)} images successfully"
            })
        else:
            return jsonify({
                "success": False,
                "error": "Failed to generate or upload any images"
            }), 500
    
    except Exception as e:
        print(f"Error: {str(e)}")
        return jsonify({
            "success": False,
            "error": f"Failed to generate batch: {str(e)}"
        }), 500

# Cài đặt Ngrok và mở tunnel cho Flask app
print("Setting up Ngrok tunnel...")
public_url = ngrok.connect(5000)
print(f"✓ Ngrok tunnel URL: {public_url}")
# đẩy url lên firebase
url = "https://vienvipvail-default-rtdb.firebaseio.com/api-graduation-image.json"
ngrok_url = public_url.public_url
response = requests.put(url, data=json.dumps(ngrok_url))

# Chạy Flask app
if __name__ == "__main__":
    print("\n" + "="*60)
    print("🚀 Starting Flask API server...")
    print("="*60)
    print(f"Current Model: {MODEL_INFO['name']}")
    print(f"Description: {MODEL_INFO['config']['description']}")
    print("\nAPI endpoints:")
    print(f"  - GET  {public_url}/health")
    print(f"  - GET  {public_url}/models  (list available models)")
    print(f"  - POST {public_url}/generate-image")
    print(f"  - POST {public_url}/generate-batch")
    print("="*60 + "\n")
    app.run(port=5000, debug=False)